# 40 Full plant model

One `solve_ivp` over concatenated 01-09 states plus BOP. Scenarios 90-99 call **this same** `PlantModel.rhs`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


In [ ]:
from plant_sim.plant import PlantModel, MODE_DISCHARGE
plant = PlantModel(P)
print("n_states", plant.n_states)
res = plant.simulate((0, 600), inputs={"mode": MODE_DISCHARGE, "P_grid_W": -400.0, "I_cell_A": P.I_discharge_100h_A}, n_eval=120)
assert np.all(np.isfinite(res["y"]))
timeseries(res["t"], {"V": res["alg"]["V_cell_V"], "I": res["alg"]["I_cell_A"], "SOC": res["alg"]["SOC"], "T": res["alg"]["T_K"]}, xlabel="t (s)", ylabel="mixed SI", title="Assembled plant discharge", callout="IA-CEL-001 / IA-SYS-005", stem="nb40")
print("nfev", res["nfev"])
